# Uttarakhand Tomato Price Prediction

End-to-end pipeline for forecasting daily tomato modal price across all 13 districts of Uttarakhand.

**Dataset:** 28,496 daily rows, 13 districts, 2020–2025
**Models:** Random Forest, XGBoost, LightGBM, combined as a simple average ensemble
**Test result (2025, held out):** R² = 0.924 · MAPE = 9.32% · MAE = ₹171.77/quintal
---

In [1]:
import subprocess, sys

required_imports = {"pandas":"pandas", "numpy":"numpy", "sklearn":"scikit-learn",
                     "xgboost":"xgboost", "lightgbm":"lightgbm", "joblib":"joblib"}
missing = []
for import_name, pip_name in required_imports.items():
    try:
        __import__(import_name)
    except ImportError:
        missing.append(pip_name)

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing, "--quiet"])
    print("Installed:", missing)
else:
    print("All libraries already installed.")


All libraries already installed.


In [2]:
import pandas as pd
import numpy as np
import warnings
import os
import joblib
import datetime

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings("ignore")
np.random.seed(42)
print("Imports ready.")

Imports ready.


## 1. Load the dataset

In [3]:
DATA_PATH = "Uttarakhand_Tomato_FINAL_Master.csv"
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Put the CSV file in the same folder as this notebook. Looking for: {DATA_PATH}")

raw = pd.read_csv(DATA_PATH)
raw["Date"] = pd.to_datetime(raw["Date"])
raw = raw.sort_values(["District", "Date"]).reset_index(drop=True)

print(f"Rows: {len(raw):,}  Districts: {raw['District'].nunique()}")
print(f"Date range: {raw['Date'].min().date()} to {raw['Date'].max().date()}")
raw.head(3)

Rows: 28,496  Districts: 13
Date range: 2020-01-01 to 2025-12-31


,Date,District,Zone,Dist_Type,Elevation_m,Num_Markets,Price_Modal_Avg,Price_Min,Price_Max,Price_Spread,...,Price_Rolling_14D_Avg,Price_Rolling_30D_Avg,Price_Volatility_7D,Price_Volatility_30D,Price_Pct_Change_1D,Price_Pct_Change_7D,Rainfall_Lag_3D,Rainfall_Lag_7D,Rainfall_Rolling_7D,Data_Source
0,2020-01-01,Almora,Kumaon_Hills,Semi_Rural,1638,2,1340,1130,1600.0,470.0,...,1340.00,1340.00,NaN,NaN,NaN,NaN,NaN,NaN,0.000,Statistically_Generated_2020
1,2020-01-02,Almora,Kumaon_Hills,Semi_Rural,1638,2,1550,1300,1950.0,650.0,...,1445.00,1445.00,148.49,148.49,15.6716,NaN,NaN,NaN,0.229,Statistically_Generated_2020
2,2020-01-03,Almora,Kumaon_Hills,Semi_Rural,1638,2,1600,1400,1790.0,390.0,...,1496.67,1496.67,137.96,137.96,3.2258,NaN,NaN,NaN,1.746,Statistically_Generated_2020


## 2. Build features

Only information that would genuinely be known before the prediction date is used:
price lags (1, 3, 7, 14, 30 days back), rolling averages and rolling standard deviation
computed on data shifted by one day first, rainfall lagged by 1 and 7 days, and calendar
features. Same-day price minimum/maximum and same-day weather are not used, since they
either directly encode the target or would not be available at the time a real
prediction is needed.

In [4]:
def build_lagged_features(frame):
    base = frame[[
        "Date", "District", "Zone", "Dist_Type", "Elevation_m", "Num_Markets",
        "Price_Modal_Avg", "Rainfall_mm", "Year", "Month", "Quarter",
        "Week_of_Year", "Day_of_Week", "Season"
    ]].copy()

    price_by_district = base.groupby("District")["Price_Modal_Avg"]

    for lag_days in (1, 3, 7, 14, 30):
        base[f"price_lag_{lag_days}d"] = price_by_district.shift(lag_days)

    for window in (7, 14, 30):
        base[f"price_rollavg_{window}d"] = price_by_district.shift(1).transform(
            lambda series: series.rolling(window, min_periods=3).mean()
        )

    for window in (7, 30):
        base[f"price_rollstd_{window}d"] = price_by_district.shift(1).transform(
            lambda series: series.rolling(window, min_periods=3).std()
        )

    base["price_pct_change_7d"] = (
        (base["price_lag_1d"] - base["price_lag_7d"]) / base["price_lag_7d"]
    ) * 100

    rainfall_by_district = base.groupby("District")["Rainfall_mm"]
    base["rainfall_lag_1d"] = rainfall_by_district.shift(1)
    base["rainfall_lag_7d"] = rainfall_by_district.shift(7)
    base["rainfall_rollavg_7d"] = rainfall_by_district.shift(1).transform(
        lambda series: series.rolling(7, min_periods=3).mean()
    )

    base["month_sin"] = np.sin(2 * np.pi * base["Month"] / 12)
    base["month_cos"] = np.cos(2 * np.pi * base["Month"] / 12)
    base["week_sin"] = np.sin(2 * np.pi * base["Week_of_Year"] / 52)
    base["week_cos"] = np.cos(2 * np.pi * base["Week_of_Year"] / 52)
    base["is_monsoon"] = base["Month"].isin([6, 7, 8, 9]).astype(int)
    base["is_winter"] = base["Month"].isin([11, 12, 1, 2]).astype(int)
    base["price_momentum_7d"] = base["price_lag_1d"] - base["price_lag_7d"]
    base["lag1_over_roll30"] = base["price_lag_1d"] / (base["price_rollavg_30d"] + 1)

    base = base.drop(columns=["Rainfall_mm"])
    base = base.dropna(subset=["price_lag_30d", "price_rollavg_30d", "price_rollstd_30d"])
    return base.reset_index(drop=True)


featured = build_lagged_features(raw)
print(f"Rows after building lag features: {len(featured):,}")
print(f"Columns: {featured.shape[1]}")
featured.head(3)

Rows after building lag features: 28,106
Columns: 35


,Date,District,Zone,Dist_Type,Elevation_m,Num_Markets,Price_Modal_Avg,Year,Month,Quarter,...,rainfall_lag_7d,rainfall_rollavg_7d,month_sin,month_cos,week_sin,week_cos,is_monsoon,is_winter,price_momentum_7d,lag1_over_roll30
0,2020-01-31,Almora,Kumaon_Hills,Semi_Rural,1638,2,1380,2020,1,1,...,0.0,4.735891,0.500000,0.866025,0.568065,0.822984,0,1,-80.0,0.975610
1,2020-02-01,Almora,Kumaon_Hills,Semi_Rural,1638,2,1110,2020,2,1,...,0.0,4.735891,0.866025,0.500000,0.568065,0.822984,0,1,290.0,1.084621
2,2020-02-02,Almora,Kumaon_Hills,Semi_Rural,1638,2,1060,2020,2,1,...,0.0,4.735891,0.866025,0.500000,0.568065,0.822984,0,1,30.0,0.882587


## 3. Encode categorical columns and split by time

In [5]:
category_columns = ["District", "Zone", "Dist_Type", "Season"]
label_encoders = {}
for column in category_columns:
    encoder = LabelEncoder()
    featured[column] = encoder.fit_transform(featured[column].astype(str))
    label_encoders[column] = encoder

predictor_columns = [c for c in featured.columns if c not in ("Date", "Price_Modal_Avg")]

train_rows = featured[featured["Year"] <= 2023]
valid_rows = featured[featured["Year"] == 2024]
test_rows  = featured[featured["Year"] == 2025]

x_train, y_train = train_rows[predictor_columns], train_rows["Price_Modal_Avg"]
x_valid, y_valid = valid_rows[predictor_columns], valid_rows["Price_Modal_Avg"]
x_test,  y_test  = test_rows[predictor_columns],  test_rows["Price_Modal_Avg"]

print(f"Train: {len(x_train):,}  Validation: {len(x_valid):,}  Test: {len(x_test):,}")
print(f"Features used: {len(predictor_columns)}")

Train: 18,603  Validation: 4,758  Test: 4,745
Features used: 33


## 4. Train the three models

Random Forest, XGBoost and LightGBM are trained independently on the same training
data and evaluated on the same held-out 2025 test year. The 2024 validation set is
only used for XGBoost and LightGBM early stopping.

In [6]:
def evaluate(label, actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    print(f"{label:16s}  MAE {mae:8.2f}   RMSE {rmse:8.2f}   R2 {r2:7.4f}   MAPE {mape:6.2f}%")
    return {"model": label, "mae": round(mae, 2), "rmse": round(rmse, 2),
            "r2": round(r2, 4), "mape": round(mape, 2)}


results = []

forest_model = RandomForestRegressor(
    n_estimators=200, max_depth=12, min_samples_leaf=6,
    max_features=0.6, n_jobs=-1, random_state=42
)
forest_model.fit(x_train, y_train)
forest_pred = forest_model.predict(x_test)
results.append(evaluate("Random Forest", y_test.values, forest_pred))

xgb_model = xgb.XGBRegressor(
    n_estimators=1200, learning_rate=0.03, max_depth=6,
    min_child_weight=8, subsample=0.8, colsample_bytree=0.75,
    reg_lambda=1.5, random_state=42, n_jobs=-1,
    early_stopping_rounds=50, eval_metric="rmse"
)
xgb_model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], verbose=False)
xgb_pred = xgb_model.predict(x_test)
results.append(evaluate("XGBoost", y_test.values, xgb_pred))

lgb_model = lgb.LGBMRegressor(
    n_estimators=1500, learning_rate=0.02, num_leaves=40,
    max_depth=7, min_child_samples=25, subsample=0.8,
    colsample_bytree=0.75, reg_lambda=1.0, random_state=42,
    n_jobs=-1, verbose=-1
)
lgb_model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
lgb_pred = lgb_model.predict(x_test)
results.append(evaluate("LightGBM", y_test.values, lgb_pred))

ensemble_pred = (forest_pred + xgb_pred + lgb_pred) / 3
results.append(evaluate("Ensemble", y_test.values, ensemble_pred))

Random Forest     MAE   171.04   RMSE   240.14   R2  0.9238   MAPE   9.25%


XGBoost           MAE   172.49   RMSE   241.62   R2  0.9229   MAPE   9.39%


LightGBM          MAE   173.22   RMSE   242.70   R2  0.9222   MAPE   9.43%
Ensemble          MAE   171.77   RMSE   240.72   R2  0.9235   MAPE   9.32%


## 5. Model comparison

In [7]:
results_df = pd.DataFrame(results).sort_values("mape").reset_index(drop=True)
results_df

,model,mae,rmse,r2,mape
0,Random Forest,171.04,240.14,0.9238,9.25
1,Ensemble,171.77,240.72,0.9235,9.32
2,XGBoost,172.49,241.62,0.9229,9.39
3,LightGBM,173.22,242.70,0.9222,9.43


## 6. Visualizations

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

NAVY, ORANGE, GREEN, STEEL, MUTED = "#1a3c5e", "#e05c2a", "#3aaa6d", "#4a6fa5", "#95a5b3"
plt.rcParams.update({"figure.facecolor": "#f7f9fb", "axes.facecolor": "#f7f9fb",
                      "axes.edgecolor": MUTED, "font.family": "DejaVu Sans",
                      "grid.color": "#e2e8ef", "grid.linewidth": 0.6})

os.makedirs("visualizations", exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Model Performance Comparison — Test Set (2025)", fontsize=13, fontweight="bold", y=1.01, color=NAVY)
order = ["Random Forest", "XGBoost", "LightGBM", "Ensemble"]
ordered = results_df.set_index("model").loc[order].reset_index()
bar_colors = [ORANGE if m == "Ensemble" else NAVY for m in ordered["model"]]
for ax, metric in zip(axes, ["mae", "rmse", "mape"]):
    bars = ax.barh(ordered["model"], ordered[metric], color=bar_colors, edgecolor="white", height=0.55)
    for bar, value in zip(bars, ordered[metric]):
        ax.text(bar.get_width() + ordered[metric].max() * 0.01, bar.get_y() + bar.get_height() / 2,
                f"{value:.2f}", va="center", fontsize=8, color=NAVY)
    unit = "Rs/qtl" if metric != "mape" else "%"
    ax.set_xlabel(f"{metric.upper()} ({unit})")
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.4)
    ax.spines[["top", "right", "left"]].set_visible(False)
plt.tight_layout()
plt.savefig("visualizations/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [9]:
dist_le = label_encoders["District"]
test_frame = pd.DataFrame({
    "Date": test_rows["Date"].values if "Date" in test_rows.columns else None,
    "District": dist_le.inverse_transform(test_rows["District"]),
    "actual": y_test.values,
    "predicted": ensemble_pred,
})

sample_districts = sorted(test_frame["District"].unique())[:6]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle("Actual vs Predicted — Sample Districts (2025 Test Set)", fontsize=13, fontweight="bold", y=1.01)
axes = axes.flatten()
for i, district in enumerate(sample_districts):
    ax = axes[i]
    subset = test_frame[test_frame["District"] == district].reset_index(drop=True)
    ax.plot(subset.index, subset["actual"], color=NAVY, linewidth=1.2, label="Actual")
    ax.plot(subset.index, subset["predicted"], color=ORANGE, linewidth=1.0, linestyle="--", label="Predicted")
    district_mae = (subset["actual"] - subset["predicted"]).abs().mean()
    ax.set_title(f"{district}  |  MAE Rs.{district_mae:.0f}", fontsize=9.5, fontweight="bold")
    ax.set_ylabel("Rs/Quintal")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.35)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("visualizations/actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()

In [10]:
importance = pd.Series(lgb_model.feature_importances_, index=predictor_columns).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [ORANGE if i < 5 else NAVY for i in range(len(importance))]
ax.barh(importance.index[::-1], importance.values[::-1], color=colors[::-1], edgecolor="white", height=0.65)
ax.set_title("Top 15 Feature Importances (LightGBM)", fontsize=12, fontweight="bold", pad=10)
ax.set_xlabel("Importance Score")
ax.grid(axis="x", alpha=0.4)
ax.spines[["top", "right", "left"]].set_visible(False)
plt.tight_layout()
plt.savefig("visualizations/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [11]:
pivot = raw.groupby(["District", "Month"])["Price_Modal_Avg"].mean().unstack()
pivot.columns = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.3, linecolor="white",
            cbar_kws={"label": "Avg Price (Rs/qtl)", "shrink": 0.7}, annot_kws={"size": 8}, ax=ax)
ax.set_title("Seasonal Price Pattern — All 13 Districts", fontsize=12, fontweight="bold", color=NAVY, pad=12)
ax.set_xlabel("Month")
ax.set_ylabel("")
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.savefig("visualizations/seasonal_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Per-district performance

In [12]:
district_rows = []
for district in sorted(test_frame["District"].unique()):
    subset = test_frame[test_frame["District"] == district]
    mae = mean_absolute_error(subset["actual"], subset["predicted"])
    r2 = r2_score(subset["actual"], subset["predicted"])
    mape = np.mean(np.abs((subset["actual"] - subset["predicted"]) / subset["actual"])) * 100
    district_rows.append({"district": district, "mae": round(mae, 1), "r2": round(r2, 3), "mape": round(mape, 2)})

district_performance = pd.DataFrame(district_rows).sort_values("mape").reset_index(drop=True)
district_performance.to_csv("district_performance.csv", index=False)
district_performance

,district,mae,r2,mape
0,Pithoragarh,174.2,0.931,8.70
1,Nainital,181.6,0.928,8.97
2,Haridwar,147.8,0.929,9.03
3,Almora,159.3,0.925,9.09
4,Rudraprayag,179.1,0.923,9.18
5,Bageshwar,160.1,0.924,9.21
6,Dehradun,204.2,0.913,9.25
7,Uttarkashi,173.3,0.919,9.34
8,UdhamSinghNagar,141.6,0.922,9.42
9,Chamoli,200.2,0.914,9.62


## 8. Save models and outputs

In [13]:
joblib.dump(forest_model, f"{MODEL_DIR}/random_forest.pkl")
joblib.dump(xgb_model, f"{MODEL_DIR}/xgboost_model.pkl")
joblib.dump(lgb_model, f"{MODEL_DIR}/lightgbm_model.pkl")
joblib.dump(label_encoders, f"{MODEL_DIR}/label_encoders.pkl")
joblib.dump(predictor_columns, f"{MODEL_DIR}/predictor_columns.pkl")

results_df.to_csv("model_comparison.csv", index=False)
test_frame.to_csv("test_predictions.csv", index=False)

print("Models saved to:", MODEL_DIR)
print("Model comparison saved to model_comparison.csv")
print("Test predictions saved to test_predictions.csv")

Models saved to: models
Model comparison saved to model_comparison.csv
Test predictions saved to test_predictions.csv


## 9. Predict a price using the trained models

This function loads the same models trained above and produces a genuine ensemble
prediction for a chosen district and date, given recent price history.

In [14]:
def season_for_month(month):
    if month in (12, 1, 2):
        return "Winter"
    if month in (3, 4):
        return "Spring"
    if month in (5, 6):
        return "Summer"
    if month in (7, 8, 9):
        return "Monsoon"
    return "Post_Monsoon"


DISTRICT_METADATA = {
    "Almora":          {"zone": "Kumaon_Hills",     "dist_type": "Semi_Rural", "elevation_m": 1638, "num_markets": 2},
    "Bageshwar":       {"zone": "Kumaon_Hills",     "dist_type": "Rural",      "elevation_m": 960,  "num_markets": 1},
    "Chamoli":         {"zone": "Garhwal_Hills",    "dist_type": "Rural",      "elevation_m": 2412, "num_markets": 1},
    "Champawat":       {"zone": "Kumaon_Hills",     "dist_type": "Rural",      "elevation_m": 1615, "num_markets": 1},
    "Dehradun":        {"zone": "Plains_Foothills", "dist_type": "Urban",      "elevation_m": 435,  "num_markets": 4},
    "Haridwar":        {"zone": "Plains",           "dist_type": "Urban",     "elevation_m": 314,  "num_markets": 5},
    "Nainital":        {"zone": "Kumaon_Hills",     "dist_type": "Semi_Urban", "elevation_m": 2084, "num_markets": 2},
    "Pauri Garhwal":   {"zone": "Garhwal_Hills",    "dist_type": "Rural",      "elevation_m": 1814, "num_markets": 1},
    "Pithoragarh":     {"zone": "Kumaon_Hills",     "dist_type": "Semi_Rural", "elevation_m": 1814, "num_markets": 2},
    "Rudraprayag":     {"zone": "Garhwal_Hills",    "dist_type": "Rural",      "elevation_m": 895,  "num_markets": 1},
    "Tehri Garhwal":   {"zone": "Garhwal_Hills",    "dist_type": "Rural",      "elevation_m": 1750, "num_markets": 2},
    "UdhamSinghNagar": {"zone": "Terai_Plains",     "dist_type": "Semi_Urban", "elevation_m": 244,  "num_markets": 7},
    "Uttarkashi":      {"zone": "Garhwal_Hills",    "dist_type": "Rural",      "elevation_m": 1158, "num_markets": 1},
}


def predict_price(district, target_date, price_lag_1d, price_lag_7d, price_lag_30d, rainfall_today):
    meta = DISTRICT_METADATA[district]
    price_lag_3d = (price_lag_1d + price_lag_7d) / 2
    price_lag_14d = (price_lag_7d + price_lag_30d) / 2
    rollavg_7d = (price_lag_1d + price_lag_7d) / 2
    rollavg_14d = (rollavg_7d + price_lag_30d) / 2
    rollavg_30d = price_lag_30d

    recent_district_rows = raw[raw["District"] == district].sort_values("Date").tail(30)
    rollstd_7d = recent_district_rows["Price_Modal_Avg"].tail(7).std()
    rollstd_30d = recent_district_rows["Price_Modal_Avg"].std()
    rainfall_lag_7d = recent_district_rows["Rainfall_mm"].tail(7).mean()
    rainfall_rollavg_7d = rainfall_lag_7d

    week_of_year = target_date.isocalendar()[1]
    quarter = (target_date.month - 1) // 3 + 1
    season_name = season_for_month(target_date.month)

    row = {
        "District": label_encoders["District"].transform([district])[0],
        "Zone": label_encoders["Zone"].transform([meta["zone"]])[0],
        "Dist_Type": label_encoders["Dist_Type"].transform([meta["dist_type"]])[0],
        "Elevation_m": meta["elevation_m"],
        "Num_Markets": meta["num_markets"],
        "Year": target_date.year,
        "Month": target_date.month,
        "Quarter": quarter,
        "Week_of_Year": week_of_year,
        "Day_of_Week": target_date.weekday(),
        "Season": label_encoders["Season"].transform([season_name])[0],
        "price_lag_1d": price_lag_1d,
        "price_lag_3d": price_lag_3d,
        "price_lag_7d": price_lag_7d,
        "price_lag_14d": price_lag_14d,
        "price_lag_30d": price_lag_30d,
        "price_rollavg_7d": rollavg_7d,
        "price_rollavg_14d": rollavg_14d,
        "price_rollavg_30d": rollavg_30d,
        "price_rollstd_7d": rollstd_7d,
        "price_rollstd_30d": rollstd_30d,
        "price_pct_change_7d": ((price_lag_1d - price_lag_7d) / price_lag_7d) * 100,
        "rainfall_lag_1d": rainfall_today,
        "rainfall_lag_7d": rainfall_lag_7d,
        "rainfall_rollavg_7d": rainfall_rollavg_7d,
        "month_sin": np.sin(2 * np.pi * target_date.month / 12),
        "month_cos": np.cos(2 * np.pi * target_date.month / 12),
        "week_sin": np.sin(2 * np.pi * week_of_year / 52),
        "week_cos": np.cos(2 * np.pi * week_of_year / 52),
        "is_monsoon": 1 if target_date.month in (6, 7, 8, 9) else 0,
        "is_winter": 1 if target_date.month in (11, 12, 1, 2) else 0,
        "price_momentum_7d": price_lag_1d - price_lag_7d,
        "lag1_over_roll30": price_lag_1d / (rollavg_30d + 1),
    }

    input_frame = pd.DataFrame([row])[predictor_columns]
    forest_out = forest_model.predict(input_frame)[0]
    xgb_out = xgb_model.predict(input_frame)[0]
    lgb_out = lgb_model.predict(input_frame)[0]
    ensemble_out = (forest_out + xgb_out + lgb_out) / 3

    print(f"District       : {district}")
    print(f"Date           : {target_date}")
    print(f"Random Forest  : Rs.{forest_out:,.0f} /qtl")
    print(f"XGBoost        : Rs.{xgb_out:,.0f} /qtl")
    print(f"LightGBM       : Rs.{lgb_out:,.0f} /qtl")
    print(f"Ensemble       : Rs.{ensemble_out:,.0f} /qtl  (Rs.{ensemble_out/100:.2f} /kg)")
    return ensemble_out


predict_price(
    district="Dehradun",
    target_date=datetime.date.today(),
    price_lag_1d=970,
    price_lag_7d=1110,
    price_lag_30d=1180,
    rainfall_today=5.0,
)

District       : Dehradun
Date           : 2026-06-26
Random Forest  : Rs.1,371 /qtl
XGBoost        : Rs.1,888 /qtl
LightGBM       : Rs.1,718 /qtl
Ensemble       : Rs.1,659 /qtl  (Rs.16.59 /kg)


np.float64(1658.9808210441176)

## 10. Open the dashboard

Two dashboards are available in this project folder:

- **tomato_dashboard.html** — static charts and metrics, opens directly in any browser, no server needed.
- **app.py** — full interactive dashboard with live prediction and multi-day forecasting, runs via Streamlit.

Run the cell below to open the static HTML dashboard directly from this notebook.

In [15]:
import webbrowser
import os

dashboard_path = os.path.abspath("tomato_dashboard.html")

if os.path.exists(dashboard_path):
    webbrowser.open(f"file://{dashboard_path}")
    print("Opened:", dashboard_path)
else:
    print("tomato_dashboard.html not found in:", os.getcwd())
    print("Make sure it is in the same folder as this notebook.")

Opened: /mnt/user-data/outputs/tomato_dashboard.html


## 11. Launch the live dashboard with prediction and forecasting

The cell below starts `app.py` as a background Streamlit server directly from
this notebook and opens it in your browser. Live prediction and multi-day
forecasting both work normally, the same as running `streamlit run app.py`
from a terminal — this cell just does that for you and keeps the notebook free
to keep working in.

Run the cell once. To stop the server later, run the following cell
(Stop the live dashboard).

In [16]:
import subprocess
import sys
import time
import webbrowser
import socket

def is_port_open(port, host="localhost"):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.5)
        return sock.connect_ex((host, port)) == 0


DASHBOARD_PORT = 8501

if is_port_open(DASHBOARD_PORT):
    print(f"A server is already running on port {DASHBOARD_PORT}.")
    print(f"Opening: http://localhost:{DASHBOARD_PORT}")
    webbrowser.open(f"http://localhost:{DASHBOARD_PORT}")
else:
    streamlit_process = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", "app.py",
         "--server.port", str(DASHBOARD_PORT), "--server.headless", "true"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

    for _ in range(20):
        if is_port_open(DASHBOARD_PORT):
            break
        time.sleep(0.5)

    if is_port_open(DASHBOARD_PORT):
        print(f"Dashboard running at http://localhost:{DASHBOARD_PORT}")
        webbrowser.open(f"http://localhost:{DASHBOARD_PORT}")
    else:
        print("Server did not start in time. Try running 'streamlit run app.py' from a terminal instead.")

Dashboard running at http://localhost:8501


### Stop the live dashboard

Run this cell to shut down the Streamlit server started above.

In [17]:
import subprocess

subprocess.run(["pkill", "-f", "streamlit run app.py"], check=False)
print("Stopped any running dashboard server on app.py.")

Stopped any running dashboard server on app.py.


Prefer the lightweight static dashboard instead? `tomato_dashboard.html`
opens directly in a browser with no server, but its Predict Price and Forecast
pages are display-only — they point you back to the live app above for real
predictions, since a static HTML file cannot call the trained models itself.